In [ ]:
import csv
import time
import os

# =====================================================================
# GENERAZIONE DATASET MASSIVO: SCENARIO 1 (1 MILIONE DI NODI)
# =====================================================================

def genera_csv_scenario1_massivo(nome_file, numero_totale_nodi=1000000):
    """
    Genera un dataset CSV massivo che modella lo Scenario 1 (Influencer Marketing).
    Implementa una scrittura a blocchi (chunking) per ottimizzare l'uso 
    della memoria RAM durante la generazione di reti su larga scala.
    """
    print(f"Inizio generazione dataset massivo: {nome_file}")
    print(f"Nodi totali: {numero_totale_nodi}")
    
    inizio = time.time()
    
    # 1. Preparazione dell'ambiente di esportazione
    cartella_destinazione = os.path.dirname(nome_file)
    if cartella_destinazione and not os.path.exists(cartella_destinazione):
        os.makedirs(cartella_destinazione)
        
    with open(nome_file, mode='w', newline='') as file_csv:
        writer = csv.writer(file_csv)
        writer.writerow(['Source', 'Target'])
        
        # 2. Inizializzazione della logica a blocchi (Chunking)
        blocco_dati = []
        dimensione_blocco = 50000 
        
        # 3. Generazione topologica: i nodi follower (3 -> N) seguono i nodi influencer (0, 1, 2)
        for follower in range(3, numero_totale_nodi):
            blocco_dati.append([follower, 0])
            blocco_dati.append([follower, 1])
            blocco_dati.append([follower, 2])
            
            # 4. Scrittura su disco e svuotamento del blocco dati per il rilascio della memoria
            if len(blocco_dati) >= dimensione_blocco * 3:
                writer.writerows(blocco_dati)
                blocco_dati.clear()
                
        # 5. Scrittura finale degli ultimi dati rimasti in memoria
        if blocco_dati:
            writer.writerows(blocco_dati)
            
    fine = time.time()
    print(f"Generazione completata con successo in {fine - inizio:.2f} secondi.")
    print(f"File salvato in: {nome_file}\n")


# =====================================================================
# ESECUZIONE SCRIPT
# =====================================================================
if __name__ == "__main__":
    N_NODI = 1000000
    NOME_FILE = '../DataSet_CasoStudio1/Rete_1M/dataset_scenario1_1MILIONE.csv'

    genera_csv_scenario1_massivo(NOME_FILE, N_NODI)


In [ ]:
import csv
import random
import time
import os

# =====================================================================
# GENERAZIONE DATASET MASSIVO: SCENARIO 2 (RETE MISTA E HUB)
# =====================================================================

def genera_csv_scenario2_massivo(nome_file, n_nodi=1000000):
    """
    Genera un dataset CSV massivo (1 milione di nodi) modellando una rete mista:
    - Influencer (Nodi 0-2) in un circolo chiuso.
    - Bot inattivi (Nodi 3 - 499.999) che seguono solo gli influencer.
    - Utenti attivi (Nodi 500.000 - 999.999) con interazioni casuali organiche.
    - Hub Strategici eletti algoritmicamente per simulare nodi ad alta centralità.
    Implementa scrittura a blocchi (chunking) per l'ottimizzazione della RAM.
    """
    print(f"Inizio generazione dataset massivo: {nome_file}")
    inizio = time.time()

    # Fissiamo il seed per garantire la riproducibilità della topologia stocastica
    random.seed(42)

    # 1. Preparazione dell'ambiente di esportazione
    cartella_destinazione = os.path.dirname(nome_file)
    if cartella_destinazione and not os.path.exists(cartella_destinazione):
        os.makedirs(cartella_destinazione)

    with open(nome_file, mode='w', newline='') as file_csv:
        writer = csv.writer(file_csv)
        writer.writerow(['Source', 'Target'])

        blocco_dati = []
        dimensione_blocco = 100000 

        def scrivi_blocco():
            """Funzione helper per il salvataggio dei dati sul disco (Chunking)."""
            nonlocal blocco_dati
            if blocco_dati:
                writer.writerows(blocco_dati)
                blocco_dati.clear()

        # 2. Topologia Influencer: Clique chiusa tra i nodi 0, 1 e 2
        blocco_dati.extend([[0, 1], [1, 2], [2, 0]])

        # 3. Generazione Bot Inattivi (Nodi 3 -> 499.999)
        print("Generazione dei Bot inattivi in corso...")
        for bot in range(3, 500000):
            blocco_dati.extend([[bot, 0], [bot, 1], [bot, 2]])
            
            if len(blocco_dati) >= dimensione_blocco:
                scrivi_blocco()

        # 4. Generazione Utenti Attivi e Hub Strategici (Nodi 500.000 -> 999.999)
        print("Generazione degli Utenti Attivi e Hub in corso...")
        start_attivi = 500000
        end_attivi = 999999

        for utente in range(start_attivi, end_attivi + 1):
            # Follow di base verso gli influencer
            blocco_dati.extend([[utente, 0], [utente, 1], [utente, 2]])

            # Distribuzione disomogenea dei link: Hub (25 link) vs Normali (2 link)
            is_hub = (utente % 500 == 0)
            num_interazioni = 25 if is_hub else 2

            # Generazione delle interazioni stocastiche intra-cluster
            for _ in range(num_interazioni):
                target_casuale = random.randint(start_attivi, end_attivi)
                if target_casuale != utente: 
                    blocco_dati.append([utente, target_casuale])

            if len(blocco_dati) >= dimensione_blocco:
                scrivi_blocco()

        # 5. Scrittura finale degli ultimi dati rimasti in memoria
        scrivi_blocco()

    fine = time.time()
    print(f"Generazione completata con successo in {fine - inizio:.2f} secondi.")
    print(f"File salvato in: {nome_file}\n")


# =====================================================================
# ESECUZIONE SCRIPT
# =====================================================================
if __name__ == "__main__":
    NOME_FILE = '../DataSet_CasoStudio1/Rete_1M/dataset_scenario2_1MILIONE.csv'
    genera_csv_scenario2_massivo(NOME_FILE)


In [ ]:
import csv
import random
import time
import os

# =====================================================================
# GENERAZIONE DATASET MASSIVO: SCENARIO 3 (TOPOLOGIA SMALL-WORLD)
# =====================================================================

def genera_csv_scenario_3_massivo(nome_file, n_nodi=1000000):
    """
    Genera un dataset CSV massivo (1 milione di nodi) modellando una community 
    organica con topologia Small-World e Preferential Attachment asimmetrico.
    Implementa interazioni locali, chiusure triadiche, bridging mirato verso 
    micro-hubs e reciprocità. Utilizza il chunking per la gestione della RAM.
    """
    print(f"Inizio generazione dataset massivo (Ottimizzato): {nome_file}")
    inizio = time.time()
    
    # Fissiamo il seed per la riproducibilità stocastica
    random.seed(42)

    # 1. Preparazione dell'ambiente di esportazione
    cartella_destinazione = os.path.dirname(nome_file)
    if cartella_destinazione and not os.path.exists(cartella_destinazione):
        os.makedirs(cartella_destinazione)

    with open(nome_file, mode='w', newline='') as file_csv:
        writer = csv.writer(file_csv)
        writer.writerow(['Source', 'Target'])

        blocco_dati = []
        dimensione_blocco = 100000 

        def scrivi_blocco():
            """Funzione helper per il salvataggio dei dati sul disco (Chunking)."""
            nonlocal blocco_dati
            if blocco_dati:
                writer.writerows(blocco_dati)
                blocco_dati.clear()

        influencers = [0, 1, 2] 
        
        # 2. Inizializzazione dei Micro-Hubs (Cluster attivi)
        micro_hubs = set(range(1000, n_nodi, 1000)) 
        micro_hubs_list = list(micro_hubs)
        
        # 3. Reciprocità: Influencer -> Micro-Hub
        print("Generazione archi di reciprocità (Influencer -> Micro-Hub)...")
        for inf in influencers:
            hub_selezionati = random.sample(micro_hubs_list, 150)
            for hub in hub_selezionati:
                blocco_dati.append([inf, hub]) 

        # 4. Rete Follower: Dinamiche Organiche e Small-World
        print("Generazione network organico (Preferential Attachment, Cluster, Triadi)...")
        for i in range(3, n_nodi):
            
            # A. Preferential Attachment Asimmetrico (Bias verso i top creator)
            if random.random() < 0.90: blocco_dati.append([i, 0]) 
            if random.random() < 0.70: blocco_dati.append([i, 1]) 
            if random.random() < 0.50: blocco_dati.append([i, 2]) 
            
            # B. Attività Locale (Micro-Hubs vs Utenti Normali)
            if i in micro_hubs:
                for j in range(1, 15):
                    if i + j < n_nodi:
                        blocco_dati.append([i, i + j]) 
            else:
                if i + 1 < n_nodi:
                    blocco_dati.append([i, i + 1]) 
                
            # C. Chiusure Triadiche (Incremento del clustering coefficient)
            if i % 3 == 0 and (i - 2) >= 3:
                blocco_dati.append([i, i - 2])
                
            # D. Bridging e Attrazione Gravitazionale (5% di probabilità)
            if random.random() < 0.05:
                if random.random() < 0.80:
                    target_casuale = random.choice(micro_hubs_list)
                else:
                    target_casuale = random.randint(3, n_nodi - 1) 
                
                if target_casuale != i: 
                    blocco_dati.append([i, target_casuale]) 
            
            # Salvataggio su disco per non saturare la memoria
            if len(blocco_dati) >= dimensione_blocco:
                scrivi_blocco()

        # 5. Scrittura finale degli ultimi dati rimasti in memoria
        scrivi_blocco()

    fine = time.time()
    print(f"Generazione Scenario Massivo completata con successo in {fine - inizio:.2f} secondi.\n")


# =====================================================================
# ESECUZIONE SCRIPT
# =====================================================================
if __name__ == "__main__":
    NOME_FILE = '../DataSet_CasoStudio1/Rete_1M/dataset_scenario3_1MILIONE.csv'
    genera_csv_scenario_3_massivo(NOME_FILE)